# 04 — Risk Categorization & Cross-Company Comparison

**Goal:** categorize each company's Risk Factors text into risk types (market, credit, operational,
regulatory, competitive, liquidity, cybersecurity) using keyword rule-matching, then build the final
cross-company comparison table combining sentiment (notebook 03) + risk categories.

## Setup

In [2]:
import os

PROJECT_ROOT = r"C:\Users\Devanshi\financial-nlp-analyzer"
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

Working directory set to: C:\Users\Devanshi\financial-nlp-analyzer


In [3]:
import os
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_000_000

EXTRACTED_DIR = "data/extracted_text"
PROCESSED_DIR = "data/processed"
COMPANIES = ["jpmorgan", "caterpillar", "accenture", "pg", "infosys"]


## Load risk text from disk

In [4]:
def load_section(company, section):
    path = os.path.join(EXTRACTED_DIR, f"{company}_{section}.txt")
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — skipping")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

risk_texts = {company: load_section(company, "risk") for company in COMPANIES}


## Risk category keyword rules

In [5]:
RISK_CATEGORIES = {
    "market_risk": ["market risk", "price volatility", "interest rate", "commodity price"],
    "credit_risk": ["credit risk", "counterparty", "default", "creditworthiness"],
    "operational_risk": ["operational risk", "business disruption", "process failure", "internal control"],
    "regulatory_risk": ["regulatory", "compliance", "legislation", "government policy"],
    "competitive_risk": ["competition", "competitive pressure", "market share"],
    "liquidity_risk": ["liquidity", "cash flow constraint", "funding"],
    "cybersecurity_risk": ["cybersecurity", "data breach", "cyber attack", "information security"],
}

def categorize_risk_sentences(risk_text):
    """Returns {category: [matching sentences]} for one company's risk text."""
    if risk_text is None:
        return {cat: [] for cat in RISK_CATEGORIES}

    doc = nlp(risk_text)
    categorized = {cat: [] for cat in RISK_CATEGORIES}

    for sent in doc.sents:
        sent_lower = sent.text.lower()
        for cat, keywords in RISK_CATEGORIES.items():
            if any(kw in sent_lower for kw in keywords):
                categorized[cat].append(sent.text.strip())
                break  # each sentence counts toward only ONE category

    return categorized


## Run categorization for every company

In [6]:
risk_category_results = {}

for company in COMPANIES:
    categorized = categorize_risk_sentences(risk_texts[company])
    risk_category_results[company] = categorized

    print(f"=== {company} ===")
    for cat, sentences in categorized.items():
        print(f"  {cat}: {len(sentences)} sentences")
    print()


=== jpmorgan ===
  market_risk: 21 sentences
  credit_risk: 21 sentences
  operational_risk: 9 sentences
  regulatory_risk: 75 sentences
  competitive_risk: 10 sentences
  liquidity_risk: 22 sentences
  cybersecurity_risk: 19 sentences

=== caterpillar ===
  market_risk: 17 sentences
  credit_risk: 4 sentences
  operational_risk: 3 sentences
  regulatory_risk: 16 sentences
  competitive_risk: 1 sentences
  liquidity_risk: 17 sentences
  cybersecurity_risk: 6 sentences

=== accenture ===
  market_risk: 0 sentences
  credit_risk: 1 sentences
  operational_risk: 9 sentences
  regulatory_risk: 30 sentences
  competitive_risk: 7 sentences
  liquidity_risk: 3 sentences
  cybersecurity_risk: 3 sentences

=== pg ===
  market_risk: 0 sentences
  credit_risk: 1 sentences
  operational_risk: 3 sentences
  regulatory_risk: 12 sentences
  competitive_risk: 4 sentences
  liquidity_risk: 3 sentences
  cybersecurity_risk: 6 sentences

=== infosys ===
  market_risk: 0 sentences
  credit_risk: 0 sentenc

## Sanity check — read a few actual matched sentences before trusting the counts

In [7]:
sample_company = COMPANIES[0]
sample_cat = "credit_risk"
sample_sentences = risk_category_results[sample_company][sample_cat][:3]

print(f"Sample '{sample_cat}' matches for {sample_company}:")
for s in sample_sentences:
    print("-", s[:200])


Sample 'credit_risk' matches for jpmorgan:
- These consequences could result in JPMorganChase experiencing increases in the allowance for credit losses, higher delinquencies, defaults and charge-offs within its commercial real estate loan portfo
- Many of these transactions expose JPMorganChase to the credit risk of its clients and counterparties, and can involve JPMorganChase in disputes and litigation if a client or counterparty defaults.
- A default by, or the financial or operational failure of, a CCP through which JPMorganChase executes contracts would require JPMorganChase to replace those contracts, thereby increasing its operationa


## Build the risk-count comparison table

In [8]:
risk_rows = []
for company in COMPANIES:
    row = {"company": company}
    row.update({
        f"riskcount_{cat}": len(sentences)
        for cat, sentences in risk_category_results[company].items()
    })
    risk_rows.append(row)

risk_df = pd.DataFrame(risk_rows)
risk_df


,company,riskcount_market_risk,riskcount_credit_risk,riskcount_operational_risk,riskcount_regulatory_risk,riskcount_competitive_risk,riskcount_liquidity_risk,riskcount_cybersecurity_risk
0,jpmorgan,21,21,9,75,10,22,19
1,caterpillar,17,4,3,16,1,17,6
2,accenture,0,1,9,30,7,3,3
3,pg,0,1,3,12,4,3,6
4,infosys,0,0,0,3,1,1,7


## Combine with sentiment scores from notebook 03 (Phase 8 — final comparison table)

In [9]:
sentiment_path = os.path.join(PROCESSED_DIR, "sentiment_scores.csv")

if os.path.exists(sentiment_path):
    sentiment_df = pd.read_csv(sentiment_path)
    comparison_df = pd.merge(sentiment_df, risk_df, on="company", how="outer")
else:
    print("sentiment_scores.csv not found — run notebook 03 first. Saving risk data only for now.")
    comparison_df = risk_df

comparison_df


,company,mdna_total_words,mdna_Negative,mdna_Negative_pct,mdna_Positive,mdna_Positive_pct,mdna_Uncertainty,mdna_Uncertainty_pct,mdna_Litigious,mdna_Litigious_pct,...,risk_Litigious_pct,risk_Constraining,risk_Constraining_pct,riskcount_market_risk,riskcount_credit_risk,riskcount_operational_risk,riskcount_regulatory_risk,riskcount_competitive_risk,riskcount_liquidity_risk,riskcount_cybersecurity_risk
0,accenture,6483,61,0.941,47,0.725,91,1.404,52,0.802,...,2.185,123,0.884,0,1,9,30,7,3,3
1,caterpillar,29091,630,2.166,187,0.643,577,1.983,270,0.928,...,1.591,115,1.040,17,4,3,16,1,17,6
2,infosys,672,5,0.744,2,0.298,10,1.488,2,0.298,...,0.112,3,0.084,0,0,0,3,1,1,7
3,jpmorgan,91,0,0.000,0,0.000,5,5.495,1,1.099,...,1.991,229,1.184,21,21,9,75,10,22,19
4,pg,13307,249,1.871,138,1.037,239,1.796,50,0.376,...,1.232,48,0.857,0,1,3,12,4,3,6


## Save the final cross-company comparison table

In [10]:
out_path = os.path.join(PROCESSED_DIR, "cross_company_comparison.csv")
comparison_df.to_csv(out_path, index=False)
print(f"Saved {out_path}")


Saved data/processed\cross_company_comparison.csv


## Quick insight check — which company leads in each category?

Run this and write 3-5 plain-English bullet points about what you see, before moving to Power BI.
This becomes the backbone of your project README.

In [11]:
pct_cols = [c for c in comparison_df.columns if c.endswith("_pct")]
riskcount_cols = [c for c in comparison_df.columns if c.startswith("riskcount_")]

print("Highest value per sentiment/risk metric:")
for col in pct_cols + riskcount_cols:
    if col in comparison_df.columns:
        top_row = comparison_df.loc[comparison_df[col].idxmax()]
        print(f"  {col}: {top_row['company']} ({top_row[col]})")


Highest value per sentiment/risk metric:
  mdna_Negative_pct: caterpillar (2.166)
  mdna_Positive_pct: pg (1.037)
  mdna_Uncertainty_pct: jpmorgan (5.495)
  mdna_Litigious_pct: jpmorgan (1.099)
  mdna_Constraining_pct: caterpillar (0.756)
  risk_Negative_pct: jpmorgan (5.879)
  risk_Positive_pct: infosys (1.684)
  risk_Uncertainty_pct: accenture (3.688)
  risk_Litigious_pct: accenture (2.185)
  risk_Constraining_pct: jpmorgan (1.184)
  riskcount_market_risk: jpmorgan (21)
  riskcount_credit_risk: jpmorgan (21)
  riskcount_operational_risk: accenture (9)
  riskcount_regulatory_risk: jpmorgan (75)
  riskcount_competitive_risk: jpmorgan (10)
  riskcount_liquidity_risk: jpmorgan (22)
  riskcount_cybersecurity_risk: jpmorgan (19)


In [12]:
risk_sentence_rows = []
for company, categorized in risk_category_results.items():
    for category, sentences in categorized.items():
        for s in sentences:
            risk_sentence_rows.append({"company": company, "risk_category": category, "sentence": s})

risk_sentences_df = pd.DataFrame(risk_sentence_rows)
risk_sentences_df.to_csv(os.path.join(PROCESSED_DIR, "risk_sentences.csv"), index=False)
print(f"Saved {len(risk_sentences_df)} risk sentences")

Saved 335 risk sentences
